In [0]:
CREATE OR REPLACE TABLE ORDERS_SALES.GOLD.ORDERS_PERFORMANCE
AS
WITH item_summary AS (
    SELECT
        order_id,
        ROUND(SUM(price + freight_value), 2) AS order_value,
        COUNT(*) AS item_count
    FROM ORDERS_SALES.SILVER.ORDER_ITEMS_ENRICHED
    GROUP BY order_id
),
payment_summary AS (
    SELECT
        order_id,
        FIRST(payment_type) AS payment_type,
        ROUND(SUM(payment_value), 2) AS total_payment,
        AVG(NULLIF(payment_installments, 0)) AS avg_installments
    FROM ORDERS_SALES.SILVER.ORDER_PAYMENTS_ENRICHED
    GROUP BY order_id
),
review_summary AS (
    SELECT
        order_id,
        ROUND(AVG(review_score), 2) AS avg_review_score,
        COUNT(*) AS review_count
    FROM ORDERS_SALES.SILVER.ORDERS_REVIEWS_ENRICHED
    GROUP BY order_id
)
SELECT 
ord.order_id,
DATE(ord.order_purchase_timestamp) AS purchse_date,
ord.order_status_group,
ord.delivery_days,
ord.delay_days,
item.order_value,
item.item_count,
payment_summary.payment_type,
payment_summary.total_payment,
ROUND(payment_summary.avg_installments, 2) AS avg_installments,
review_summary.avg_review_score
FROM
ORDERS_SALES.SILVER.ORDERS_ENRICHED AS ord 
LEFT JOIN item_summary AS item ON ord.order_id = item.order_id
LEFT JOIN payment_summary AS payment_summary ON ord.order_id = payment_summary.order_id 
LEFT JOIN review_summary AS review_summary ON ord.order_id = review_summary.order_id;

In [0]:
-- VALIDATE ORDERS_SALES.GOLD.ORDERS_PERFORMANCE
SELECT
*
FROM
ORDERS_SALES.GOLD.ORDERS_PERFORMANCE
WHERE total_payment IS NOT NULL OR avg_review_score IS NOT NULL
LIMIT 15;